# Create, edit and delete records
The whole life of a record through the API: create a hub, a work and an instance of it, edit them, look at their versions, then delete them. Use it to try a change to the API or to reproduce a cataloging bug.

**Local stack only.** It creates records and deletes them again at the end, so it stops on dev and stage, which are shared. Signing in uses your stack's API login (`API_KEYCLOAK_USER`, from the stack checkout), with the API's `create` and `update` roles.

In [ ]:
import json, uuid
from bluecore import show, target
bc = target()
bc.require_writable()               # stops here on dev and stage
AUTH = bc.auth_headers()            # a Keycloak token for your stack's API login
LD = {**AUTH, "Content-Type": "application/ld+json"}   # a raw JSON-LD body
RUN = uuid.uuid4().hex[:8]          # marks this run's records: https://example.org/notebook/<RUN>/...
bc

## Create
The bodies start from the API's own request examples (`bc.example`). The API mints the record's URI and UUID itself; the `@id` you send just has to be unique.

Create takes either raw JSON-LD (`Content-Type: application/ld+json`) or the Sinopia body (`application/vnd.sinopia+json`, `{"data": "<JSON-LD as a string>"}`), which can also name the instance's work by its database id.

In [ ]:
hub = {**bc.example("/hubs/"), "@id": f"https://example.org/notebook/{RUN}/hub"}
r = bc.post("/api/hubs/", json=hub, headers=LD); show(r, 300)
hub = r.json()

work = {**bc.example("/works/"), "@id": f"https://example.org/notebook/{RUN}/work"}
r = bc.post("/api/works/", json=work, headers=LD); show(r, 300)
work = r.json()

instance = {**bc.example("/instances/"), "@id": f"https://example.org/notebook/{RUN}/instance"}
body = {"data": json.dumps(instance), "work_id": work["id"]}
r = bc.post("/api/instances/", json=body, headers={**AUTH, "Content-Type": "application/vnd.sinopia+json"}); show(r, 300)
instance = r.json()

print("\nhub", hub["uuid"], "· work", work["uuid"], "· instance", instance["uuid"], "of work id", instance["work_id"])

> **Known issue:** a new work that points at an existing Blue Core hub (`"bf:expressionOf": {"@id": "<hub URI>"}`) fails with a `500`: the API tries to store the hub again as an "other resource" and hits the unique URI constraint. That's why the work above isn't linked to the hub. <!-- found 2026-10-16, bluecore_api 0.35.2 -->

## Edit
A PUT sends the whole record. Read it as JSON-LD, change it, send it back.

In [ ]:
current = bc.get(f"/api/works/{work['uuid']}.jsonld").json()
current["title"] = [{"@type": ["Title"], "mainTitle": ["Pride and prejudice (second edit)"]}]
r = bc.put(f"/api/works/{work['uuid']}", json=current, headers=LD)
show(r, 400)

## Versions
Each save is a version, with who made it.

In [ ]:
versions = bc.get(f"/api/works/{work['uuid']}/versions").json()["versions"]
for v in versions:
    old = bc.get(f"/api/works/{work['uuid']}/version/{v['id']}").json()
    print(v["timestamp"], v["user"], "→", old["data"]["title"][0]["mainTitle"])

## Delete
Instances first, then the work and the hub. Deleting a work also deletes its instances. The vocabulary terms they pointed at (language, content type, role) stay, as other resources, for other records to share.

In [ ]:
for kind, uuid_ in [("instances", instance["uuid"]), ("works", work["uuid"]), ("hubs", hub["uuid"])]:
    r = bc.delete(f"/api/{kind}/{uuid_}", headers=AUTH)
    print(kind, uuid_, r.status_code)
    assert bc.get(f"/api/{kind}/{uuid_}.jsonld").status_code == 404
print("All gone.")